# PQ — Avg relative error vs Bits per vector

Single **y** vs **x** pair from the main metrics loop (`plot_relerr_vs_x`). Legend grouping follows the source notebook for that x-axis (for example, curves grouped by $N_{subq}$ when x is bits per subquantizer).

Same plot from the shell (optional `--run-recon-eval` for distortion): `python3 scripts/figure_generators/pq_xy_figures.py --metric relerr --x bits_per_vector --data-dir <path-to-relerr_cpp> [--run-recon-eval] --no-show`


In [ ]:
# =============================================================================
# CONFIG — edit USER_* assignments, then Run All.
# Any USER_* you set to a non-None value overrides the default in the data cell below.
# Leave USER_* as None to keep that default (e.g. USER_X_COLUMNS_TO_PLOT = None → use full X_COLUMNS_TO_PLOT).
# Cell order: this CONFIG → imports/paths → data & helpers → plots.
# =============================================================================

from pathlib import Path

# Paths (optional overrides)
USER_DATA_DIR = None  # e.g. Path("/home/.../results/relerr_cpp")
USER_PQ_FAISS_ADC_SUMMARY = None  # e.g. Path(".../pq_faiss_adc_summary.csv")

# Who to plot (all datasets in one go)
USER_METHODS_TO_PLOT = ["PQ"]
USER_DATASETS_TO_PLOT = ["deep", "bigann", "gist", "msmarco", "openai"]

# Pin one main-loop figure (same keys as Y_METRICS in the data cell)
TARGET_Y_METRIC = 'relerr'  # relerr | spearman | recall | recon_error (Distortion error → recon_error)
TARGET_X_COL = 'bits_per_vector'  # e.g. n_subquantizers, nbits, bits_per_vector, adc_cpu_time_pp / adc_cpu_time_pp_ms

USER_Y_METRICS = None  # xy split: use TARGET_Y_METRIC (plot cell pins the sweep)

# Per-x-column y limits or None for auto
USER_YLIM_CONFIG = {}

# Subquantizer filters (same dict structure as the data cell defaults)
USER_NBITS_PLOT_SUBQUANTIZERS = None
USER_NUM_SUBQ_PLOT_SUBQUANTIZERS = None

# Grouping / ADC time unit
USER_BITS_PER_VECTOR_GROUPING = None
USER_COMPRESSION_RATE_GROUPING = None
USER_ADC_TIME_UNIT = None

# X columns for the main sweep (list of (column, label) tuples) — None keeps data-cell default
USER_X_COLUMNS_TO_PLOT = None

USER_ADDITIONAL_PLOTS = None
USER_BAR_PLOTS = None



In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator, LogLocator, ScalarFormatter

# --- Plot style (aligned with DARTH_plus_conformal.ipynb) ---
# Base style - will be overridden to font.size=40 for individual plots
plt.rcParams.update({
    "font.size": 20,
    "axes.titlesize": 40,
    "axes.labelsize": 20,
    "xtick.labelsize": 17,
    "ytick.labelsize": 17,
    "legend.fontsize": 21,
})
# plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

DATA_DIR = Path("/data/cpanourg/2-hdvc/results/relerr_cpp")
if not DATA_DIR.exists():
    DATA_DIR = Path("/home/cpanourg/projects/2-hdvc/results/relerr_cpp")
CSV_PATTERN = "*_adc_vs_exact_eval.csv"

# Completed PQ FAISS ADC dense-distance/timing export.
# This supersedes the older PQ timing columns from DATA_DIR for PQ plots.
PQ_FAISS_ADC_SUMMARY = Path("/mnthdd/cpanourg/2-hdvc/results/pq/pq_faiss_adc_summary.csv")
PQ_FAISS_FIGURES_DIR = PQ_FAISS_ADC_SUMMARY.parent / "figures"


# Canonical axis labels for figures (CSV columns remain `nbits`, `n_subquantizers`).
N_SUBQ_X_AXIS_LABEL = r"# of subquantizers ($N_{subq}$)"
NBITS_X_AXIS_LABEL = "Bits per subq (bps)"
LEGEND_BPS_PREFIX = "bps="

FIG_AVG_RELERR_Y_LABEL = "Avg relative error"
FIG_DISTORTION_Y_LABEL = "Distortion error"
FIG_SRC_Y_LABEL = "SRC"


def curve_legend_label(group_col: str, group_val) -> str:
    """Legend text for a curve keyed by nbits or n_subquantizers."""
    if group_col == "nbits":
        return f"{LEGEND_BPS_PREFIX}{int(group_val)}"
    return rf"$N_{{subq}}={int(group_val)}$"


def nudge_bits_per_subq_xlabel(ax) -> None:
    """Shift the x-axis title slightly to the right for the Bps x-axis label."""
    if ax.get_xlabel().strip().casefold() != NBITS_X_AXIS_LABEL.casefold():
        return
    lx, ly = ax.xaxis.label.get_position()
    ax.xaxis.label.set_position((lx + 0.035, ly))




In [ ]:
# ----- Optional path overrides (USER_* from CONFIG cell) -----
if "USER_DATA_DIR" in globals() and USER_DATA_DIR is not None:
    DATA_DIR = Path(USER_DATA_DIR)
if "USER_PQ_FAISS_ADC_SUMMARY" in globals() and USER_PQ_FAISS_ADC_SUMMARY is not None:
    PQ_FAISS_ADC_SUMMARY = Path(USER_PQ_FAISS_ADC_SUMMARY)
    PQ_FAISS_FIGURES_DIR = PQ_FAISS_ADC_SUMMARY.parent / "figures"

from glob import glob

def load_relerr_data(data_dir: Path, pattern: str = CSV_PATTERN) -> pd.DataFrame:
    """Load and concatenate all relative-error CSVs in `data_dir`.

    Assumes filenames of the form `{dataset}_{method}_adc_vs_exact_eval.csv`.
    """
    csv_paths = sorted(data_dir.glob(pattern))
    if not csv_paths:
        raise FileNotFoundError(f"No CSVs matching {pattern!r} found in {data_dir}")

    dfs = []
    for path in csv_paths:
        df = pd.read_csv(path)

        # Infer dataset/method from filename if needed
        name = path.stem  # e.g., deep_OPQ_adc_vs_exact_eval
        parts = name.split("_")
        if len(parts) >= 2:
            file_dataset, file_method = parts[0], parts[1]
            if "dataset" not in df.columns:
                df["dataset"] = file_dataset
            if "method" not in df.columns:
                df["method"] = file_method
        dfs.append(df)

    all_df = pd.concat(dfs, ignore_index=True)

    # Ensure expected columns exist
    required_cols = [
        "method",
        "dataset",
        "n_subquantizers",
        "nbits",
        "bits_per_vector",
        "rel_error_mean",
        "rel_error_std",
    ]
    missing = [c for c in required_cols if c not in all_df.columns]
    if missing:
        raise ValueError(f"Missing columns in concatenated DF: {missing}")

    return all_df


def _load_eval_csvs(data_dir: Path, pattern: str, suffix: str):
    """Load and concatenate CSVs matching pattern. Returns None if no files found."""
    csv_paths = sorted(data_dir.glob(pattern))
    if not csv_paths:
        return None
    dfs = []
    for path in csv_paths:
        df = pd.read_csv(path)
        name = path.stem.replace(suffix, "")
        parts = name.split("_")
        if len(parts) >= 2:
            if "dataset" not in df.columns:
                df["dataset"] = parts[0]
            if "method" not in df.columns:
                df["method"] = parts[1]
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)



def load_pq_faiss_adc_summary(summary_path: Path = PQ_FAISS_ADC_SUMMARY) -> pd.DataFrame:
    """Load completed PQ FAISS ADC timing + dense relative-error summary.

    Normalizes timing to the notebook's expected columns:
    - adc_cpu_time_pp: seconds per (query, db) pair
    - adc_cpu_time_pp_ms: milliseconds per pair
    - adc_time_s: total measured FAISS ADC time for the query batch
    """
    summary_path = Path(summary_path)
    if not summary_path.exists():
        raise FileNotFoundError(f"PQ FAISS ADC summary not found: {summary_path}")

    df = pd.read_csv(summary_path)
    df["method"] = df.get("method", "PQ")
    df["method"] = df["method"].fillna("PQ")

    if "per_pair_adc_time_ns" in df.columns:
        df["adc_cpu_time_pp"] = df["per_pair_adc_time_ns"] * 1e-9
        df["adc_cpu_time_pp_ms"] = df["per_pair_adc_time_ns"] * 1e-6
    if "per_pair_adc_time_ns_std" in df.columns:
        df["adc_cpu_time_pp_std"] = df["per_pair_adc_time_ns_std"] * 1e-9
        df["adc_cpu_time_pp_ms_std"] = df["per_pair_adc_time_ns_std"] * 1e-6

    # Keep the historical column names usable for existing plotting cells.
    if "adc_time_s" not in df.columns and "faiss_adc_time_s" in df.columns:
        df["adc_time_s"] = df["faiss_adc_time_s"]

    return df


def replace_pq_with_faiss_adc_summary(relerr_df: pd.DataFrame, summary_path: Path = PQ_FAISS_ADC_SUMMARY) -> pd.DataFrame:
    """Use FAISS-exported PQ rows when available, preserving non-PQ methods from relerr_df."""
    if not Path(summary_path).exists():
        print(f"⚠️  {summary_path} not found — using PQ rows from {DATA_DIR}")
        return relerr_df

    pq_df = load_pq_faiss_adc_summary(summary_path)
    non_pq_df = relerr_df[relerr_df["method"] != "PQ"].copy()
    combined = pd.concat([non_pq_df, pq_df], ignore_index=True, sort=False)
    print(f"Using PQ FAISS ADC summary: {summary_path} ({len(pq_df)} PQ rows)")
    return combined

def build_plot_df(relerr_df: pd.DataFrame, data_dir: Path) -> pd.DataFrame:
    """Build unified dataframe with all metrics: relerr, compression_rate, spearman, recall, recon_error.

    Starts with relerr_df (has rel_error_mean) and merges in eval CSVs.
    Merges on (dataset, method, experiment_folder, n_subquantizers, nbits).
    """
    plot_df = relerr_df.copy()
    merge_cols = ["dataset", "method", "n_subquantizers", "nbits"]
    if "experiment_folder" in plot_df.columns:
        merge_cols.append("experiment_folder")

    for suffix, metric_cols in [
        ("_compression_rate", ["compression_rate"]),
        ("_reconstruction_error", ["reconstruction_error"]),
        ("_spearman", ["spearman"]),
        ("_recall", ["recall_1", "recall_10", "recall_100"]),
    ]:
        extra = _load_eval_csvs(data_dir, f"*{suffix}.csv", suffix)
        if extra is not None:
            add_cols = [c for c in metric_cols if c in extra.columns]
            if add_cols:
                cols = [c for c in merge_cols if c in extra.columns] + add_cols
                plot_df = plot_df.merge(extra[cols].drop_duplicates(), on=merge_cols, how="left")

    return plot_df


relerr_df = replace_pq_with_faiss_adc_summary(load_relerr_data(DATA_DIR), PQ_FAISS_ADC_SUMMARY)
relerr_df.head()

# ============================================================================
# CONFIGURATION: Select methods, datasets, and x-axis columns to plot
# ============================================================================

# Select which methods to plot (available: OPQ, PQ)
METHODS_TO_PLOT = ["PQ"]  # Change to ["OPQ"] or ["PQ"] to plot only one

# Select which datasets to plot (available: deep, gist)
DATASETS_TO_PLOT = ["deep", "bigann", "gist", "msmarco", "openai"]  # Change to ["deep"] or ["gist"] to plot only one
# DATASETS_TO_PLOT = ["deep"]  # Change to ["deep"] or ["gist"] to plot only one

# ADC CPU time unit: "ms" for milliseconds, "s" for seconds
ADC_TIME_UNIT = "s"

_adc_col = "adc_cpu_time_pp_ms" if ADC_TIME_UNIT == "ms" else "adc_cpu_time_pp"
_adc_label = f"ADC time ({ADC_TIME_UNIT})"

# Select which x-axis columns to plot (column_name, display_label)
# Available columns: n_subquantizers, nbits, bits_per_vector, train_time_s, or any other numeric column
X_COLUMNS_TO_PLOT = [
    ("n_subquantizers", N_SUBQ_X_AXIS_LABEL),
    ("nbits", NBITS_X_AXIS_LABEL),
    ("bits_per_vector", "Bits per vector"),
    # ("train_time_s", "Training time (seconds)"),  # Avg relative error vs training time
    # ("adc_time_s", "ADC time (seconds)"),  # Avg relative error vs asymmetric distance computation time
    # ("distance_table_time_s", "Distance table time (seconds)"),  # Avg relative error vs distance table computation time
    # ("adc_time_2", "ADC time (s)"),  # distance_table_time_s + adc_time_s (wall clock)
    # ("adc_cpu_time", "ADC CPU time (s)"),  # total ADC CPU time (process_time)
    # ("adc_cpu_time_pp", "ADC CPU time per pair (s)"),  # per (query, db) pair
    (_adc_col, _adc_label),
]

# For nbits x-axis: which n_subquantizers (curves) to show per dataset.
# - dict: dataset name -> list of subq (e.g. {"deep": [1, 2, 4, 8, 16], "gist": [1, 4, 8]})
# - list: same list for all datasets (e.g. [1, 2, 4, 8, 16])
# - None: show all M for all datasets
NBITS_PLOT_SUBQUANTIZERS = {"deep": [1, 4, 12, 24, 32, 96], "bigann": [1, 4, 16, 32, 64, 128], "gist": [1, 8, 40, 60, 320, 960], "msmarco": [1, 8, 32, 64, 256, 1024], "openai": [1, 32, 128, 256, 512, 1536]}  # or [1, 2, 4, 8] or None

# For M x-axis: which n_subquantizers (x-axis points) to show per dataset.
# - dict: dataset name -> list of subq; list: same for all; None: show all
NUM_SUBQ_PLOT_SUBQUANTIZERS = {"deep": [1, 4, 12, 24, 32, 96], "bigann": [1, 4, 16, 32, 64, 128], "gist": [1, 8, 40, 60, 320, 960], "msmarco": [1, 8, 32, 64, 256, 1024], "openai": [1, 32, 128, 256, 512, 1536]}  # or [1, 2, 4, 8] or None

# You can also add custom columns, e.g.:
# X_COLUMNS_TO_PLOT = [
#     ("n_subquantizers", "M"),
#     ("train_time_s", "Training time (seconds)"),
# ]

# Optional: Set y-axis limits for common x-axis columns
# Format: {column_name: (ymin, ymax)} or {column_name: None} to use auto
# This ensures all plots with the same x-axis have the same y-axis range for easy comparison
YLIM_CONFIG = {
    # "n_subquantizers": (0.0, 0.6),  # Example: set ylim for M plots
    # "nbits": (0.0, 0.3),             # Example: set ylim for nbits plots
    # "bits_per_vector": None,         # Use None or omit to use auto scaling
}

# Grouping configuration for bits_per_vector plots
# Options: "nbits" or "n_subquantizers"
# - "nbits": Groups by bits per subvector (shows how precision per subvector affects performance)
# - "n_subquantizers": Groups by M (shows trade-off between granularity and total bits)
BITS_PER_VECTOR_GROUPING = "n_subquantizers" # "n_subquantizers"  # Change to "nbits" to group by bits per subvector instead

# Additional plots with different y-axis columns
# Format: list of tuples (x_col, x_label, y_col, y_label, group_by)
# - x_col: Column name for x-axis
# - x_label: Display label for x-axis
# - y_col: Column name for y-axis (e.g., "adc_time_s", "rel_error_mean", "train_time_s")
# - y_label: Display label for y-axis (or None for auto-generated)
# - group_by: "nbits" or "n_subquantizers" - how to group the curves
ADDITIONAL_PLOTS = [
    # Plot ADC time vs M, grouped by nbits
    ("n_subquantizers", N_SUBQ_X_AXIS_LABEL, _adc_col, _adc_label, "nbits"),
    # Add more plots here as needed
    # ("bits_per_vector", "Bits per vector", "adc_time_s", "ADC time (seconds)", "nbits"),
    
    # Training time plots
    # ("n_subquantizers", "M", "train_time_s", "Training time (seconds)", "nbits"),
    # ("nbits", "nbits", "train_time_s", "Training time (seconds)", "n_subquantizers"),
    # ("bits_per_vector", "Bits per vector", "train_time_s", "Training time (seconds)", "nbits"),
]

# Bar plots for timing metrics (bin-style plots) with averaging
# Format: list of tuples (x_col, x_label, y_col, y_label, group_by)
# - x_col: Column name for x-axis (will be grouped into bins/bars)
# - x_label: Display label for x-axis
# - y_col: Column name for y-axis (e.g. "train_time_s", "adc_time_s", "distance_table_time_s")
# - y_label: Display label for y-axis (or None for auto-generated)
# - group_by: "nbits" or "n_subquantizers" - dimension to average over (the legend dimension)
BAR_PLOTS = [
    # Training time bar plots (averaged over legend dimension)
    # Plot 1: Training time vs M (averaged over nbits) - single bars per M
    # ("n_subquantizers", "M", "train_time_s", "Training time (seconds)", "nbits"),
    # Plot 2: Training time vs nbits (averaged over M) - single bars per nbits
    # ("nbits", "nbits", "train_time_s", "Training time (seconds)", "n_subquantizers"),

    # ADC time bar plots (averaged over legend dimension)
    # Plot 3: ADC time vs M (averaged over nbits)
    # ("n_subquantizers", "M", "adc_time_s", "ADC time (seconds)", "nbits"),
    # Plot 4: ADC time vs nbits (averaged over M)
    # ("nbits", "nbits", "adc_time_s", "ADC time (seconds)", "n_subquantizers"),

    # Distance table time bar plots (averaged over legend dimension)
    # Plot 5: Distance table time vs M (averaged over nbits)
    # ("n_subquantizers", "M", "distance_table_time_s", "Distance table time (seconds)", "nbits"),
    # Plot 6: Distance table time vs nbits (averaged over M)
    # ("nbits", "nbits", "distance_table_time_s", "Distance table time (seconds)", "n_subquantizers"),
]

# Y-axis metrics to plot (applies to ALL plots: relerr vs nbits, vs M, compression_rate vs Y, etc.)
# Options: relerr, spearman, recon_error, recall (or recall_1, recall_10, recall_100)
Y_METRICS = ["relerr"]  # e.g. ["relerr", "spearman", "recall", "recon_error"]

# Metric name -> (column name, display label)
Y_METRIC_MAP = {
    "relerr": ("rel_error_mean", FIG_AVG_RELERR_Y_LABEL),
    "recon_error": ("reconstruction_error", FIG_DISTORTION_Y_LABEL),
    "spearman": ("spearman", FIG_SRC_Y_LABEL),
    "recall_1": ("recall_1", "Recall@1"),
    "recall_10": ("recall_10", "Recall@10"),
    "recall_100": ("recall_100", "Recall@100"),
}

# For compression_rate vs Y plots only: how to group curves
COMPRESSION_RATE_GROUPING = "nbits"  # or "n_subquantizers"

# Build unified plot_df with all metrics merged in
plot_df = build_plot_df(relerr_df, DATA_DIR)

# ----- Apply USER_* overrides from the CONFIG cell (run before this cell) -----
# None means "keep the default from this cell" — only non-None USER_* values override.
for _k in (
    "METHODS_TO_PLOT",
    "DATASETS_TO_PLOT",
    "Y_METRICS",
    "YLIM_CONFIG",
    "NBITS_PLOT_SUBQUANTIZERS",
    "NUM_SUBQ_PLOT_SUBQUANTIZERS",
    "BITS_PER_VECTOR_GROUPING",
    "COMPRESSION_RATE_GROUPING",
    "ADC_TIME_UNIT",
    "X_COLUMNS_TO_PLOT",
    "ADDITIONAL_PLOTS",
    "BAR_PLOTS",
):
    _uk = "USER_" + _k
    if _uk in globals() and globals()[_uk] is not None:
        globals()[_k] = globals()[_uk]


# Prefer CPU-time columns when available (isolates from system load).
# Run: python scripts/evals/measure_adc_cpu_time.py  to populate these.
if "adc_time_cpu_s" in plot_df.columns:
    plot_df["adc_time_s"] = plot_df["adc_time_cpu_s"].combine_first(plot_df["adc_time_s"])
if "distance_table_time_cpu_s" in plot_df.columns:
    plot_df["distance_table_time_s"] = plot_df["distance_table_time_cpu_s"].combine_first(plot_df["distance_table_time_s"])
if "adc_cpu_time" in plot_df.columns:
    plot_df["adc_time_2"] = plot_df["adc_cpu_time"].combine_first(plot_df["adc_time_2"])

# Derive millisecond per-pair column (avoids scientific notation in plots)
if "adc_cpu_time_pp" in plot_df.columns:
    plot_df["adc_cpu_time_pp_ms"] = plot_df["adc_cpu_time_pp"] * 1e3


In [ ]:
# Color and marker palettes (will be dynamically assigned based on data)
# These are ordered lists that will be cycled through based on unique values in the data
COLOR_PALETTE = [
    "tab:blue", "tab:green", "tab:purple", "tab:orange", "tab:red",
    "tab:brown", "tab:pink", "tab:gray", "tab:olive", "tab:cyan",
    "tab:blue", "tab:green", "tab:purple", "tab:orange", "tab:red"
]

MARKER_PALETTE = [
    "o", "v", "s", "^", "D", "<", ">", "p", "*", "h", "H", "X", "d", "P", "8"
]

def create_dynamic_color_map(unique_values):
    """Create a color map dynamically based on unique values in the data."""
    sorted_values = sorted(unique_values)
    color_map = {}
    for i, val in enumerate(sorted_values):
        color_map[val] = COLOR_PALETTE[i % len(COLOR_PALETTE)]
    return color_map

def create_dynamic_marker_map(unique_values):
    """Create a marker map dynamically based on unique values in the data."""
    sorted_values = sorted(unique_values)
    marker_map = {}
    for i, val in enumerate(sorted_values):
        marker_map[val] = MARKER_PALETTE[i % len(MARKER_PALETTE)]
    return marker_map


def plot_relerr_vs_x(
    df: pd.DataFrame, 
    x_col: str, 
    x_label: str, 
    y_col: str = "rel_error_mean",
    y_label: str = None,
    methods: list = None,
    datasets: list = None,
    ylim: tuple = None,
    group_by: str = None,
    output_dir: Path = None,
    nbits_subquantizers=None,
    num_subq_plot_subquantizers=None,
):
    """
    Plot `y_col` vs `x_col`, with separate figures per method and dataset.

    Grouping logic:
    - When x_col is 'n_subquantizers': separate curves by nbits.
    - When x_col is 'nbits': separate curves by n_subquantizers.
    - For x_col 'nbits', nbits_subquantizers restricts which M curves are shown: None = all;
      list = same M for all datasets; dict = dataset -> list of subq per dataset.
    - For x_col 'n_subquantizers', num_subq_plot_subquantizers restricts which M values are shown: same format.
    - When x_col is 'bits_per_vector': grouping is controlled by BITS_PER_VECTOR_GROUPING config
      (can be "nbits" or "n_subquantizers").
    
    Args:
        df: DataFrame with results
        x_col: Column name to plot on x-axis
        x_label: Display label for x-axis
        y_col: Column name to plot on y-axis (default: "rel_error_mean")
        y_label: Display label for y-axis (if None, auto-generated from y_col)
        methods: List of methods to plot (if None, uses all available)
        datasets: List of datasets to plot (if None, uses all available)
        ylim: Optional tuple (ymin, ymax) to set y-axis limits (if None, uses auto)
        group_by: Optional override for grouping ("nbits" or "n_subquantizers"). If None, uses default logic.
        output_dir: Directory to save plots
    """
    if methods is None:
        methods = sorted(df["method"].unique())
    if datasets is None:
        datasets = sorted(df["dataset"].unique())
    
    if output_dir is None:
        output_dir = Path("./../../experiments/plots/relerr_cpp")
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Determine grouping variable based on x_col (or use group_by if provided)
    if group_by is not None:
        # Override default grouping with explicit group_by parameter.
        group_aliases = {
            "nbits": "nbits",
            "bits": "nbits",
            "n_subquantizers": "n_subquantizers",
            "M": "n_subquantizers",
            "subq": "n_subquantizers",
        }
        group_col = group_aliases.get(group_by)
        if group_col is None:
            raise ValueError(f"group_by must be 'nbits' or 'n_subquantizers', got {group_by}")
    elif x_col == "n_subquantizers":
        group_col = "nbits"
    elif x_col == "nbits":
        group_col = "n_subquantizers"
    elif x_col == "bits_per_vector":
        # Use configuration to determine grouping
        if BITS_PER_VECTOR_GROUPING == "nbits":
            group_col = "nbits"
        else:  # default to n_subquantizers
            group_col = "n_subquantizers"
    else:
        # For other columns, default to grouping by nbits
        group_col = "nbits"
    
    for method in methods:
        for dataset in datasets:
            sub = df[(df["method"] == method) & (df["dataset"] == dataset)].copy()
            if sub.empty:
                continue
            # For nbits x-axis: optionally restrict which n_subquantizers (curves) to show (per dataset or global)
            if x_col == "nbits" and nbits_subquantizers is not None:
                subq_list = nbits_subquantizers.get(dataset) if isinstance(nbits_subquantizers, dict) else nbits_subquantizers
                if subq_list is not None:
                    sub = sub[sub["n_subquantizers"].isin(subq_list)]
            # For n_subquantizers x-axis: optionally restrict which M values to show (per dataset or global)
            if x_col == "n_subquantizers" and num_subq_plot_subquantizers is not None:
                subq_list = num_subq_plot_subquantizers.get(dataset) if isinstance(num_subq_plot_subquantizers, dict) else num_subq_plot_subquantizers
                if subq_list is not None:
                    sub = sub[sub["n_subquantizers"].isin(subq_list)]
            if sub.empty:
                continue

            req_cols = (x_col, y_col, group_col)
            missing = [c for c in req_cols if c not in sub.columns]
            if missing:
                print(
                    f"Skipping {method}/{dataset}: missing columns {missing!r} "
                    f"(need {x_col!r}, {y_col!r}, {group_col!r})"
                )
                continue
            row_ok = sub[list(req_cols)].notna().all(axis=1)
            if not row_ok.any():
                print(
                    f"Skipping {method}/{dataset}: no rows with non-null "
                    f"{x_col!r}, {y_col!r}, and {group_col!r} "
                    f"(y values may be missing in eval CSVs)."
                )
                continue
            sub = sub.loc[row_ok]

            # Create dynamic color and marker maps based on actual data values
            unique_group_values = sorted(sub[group_col].unique())
            color_map = create_dynamic_color_map(unique_group_values)
            marker_map = create_dynamic_marker_map(unique_group_values)
            
            # Create figure (using default size from rcParams)
            fig, ax = plt.subplots()
            
            # Group by the grouping column and plot each group
            grouped = sub.groupby(group_col)
            for group_val, group_df in grouped:
                # Sort by x_col for proper line plotting
                group_df_sorted = group_df.sort_values(x_col)
                
                # Get color and marker for this group
                color = color_map[group_val]
                marker = marker_map[group_val]
                
                # Plot line without error bars
                ax.plot(
                    group_df_sorted[x_col],
                    group_df_sorted[y_col],
                    label=curve_legend_label(group_col, group_val),
                    color=color,
                    marker=marker,
                    markersize=12,
                    linewidth=2,
                    markeredgewidth=2,
                    markeredgecolor="black",
                )
            
            ax.set_xlabel(x_label, fontsize=40)
            # Set y-axis label - use provided y_label or generate from y_col
            if y_label is None:
                if y_col == "rel_error_mean":
                    y_label = FIG_AVG_RELERR_Y_LABEL
                elif y_col == "adc_time_s":
                    y_label = "ADC time (seconds)"
                elif y_col == "train_time_s":
                    y_label = "Training time (seconds)"
                elif y_col == "distance_table_time_s":
                    y_label = "Distance table time (seconds)"
                elif y_col == "encoding_time_s":
                    y_label = "Encoding time (seconds)"
                else:
                    y_label = y_col.replace("_", " ").title()
            ax.set_ylabel(y_label, fontsize=40)
            if y_col == "rel_error_mean" and x_col == "nbits":
                # Avg relative error vs Bps (curves = N_subq): pull y-label left of y-tick numerals.
                ax.yaxis.set_label_coords(-0.22, 0.38)
            elif y_col == "rel_error_mean" and x_col == "n_subquantizers":
                ax.yaxis.set_label_coords(-0.14, 0.38)
            # No title - method and dataset are specified as inputs
            ax.tick_params(labelsize=39)
            if ADC_TIME_UNIT == "s":
                ax.ticklabel_format(style="scientific", axis="both", scilimits=(-3, 3))
                for axis in (ax.xaxis, ax.yaxis):
                    axis.offsetText.set_fontsize(27)
                ax.xaxis.offsetText.set_position((1.05, 0))
            
            # Set y-axis limits if specified
            if ylim is not None:
                ax.set_ylim(ylim)
            adc_time_cols = {"adc_time_s", "adc_time_2", "adc_cpu_time", "adc_cpu_time_pp", "adc_cpu_time_pp_ms"}
            if x_col in adc_time_cols:
                ax.set_xscale("log")
            if y_col in adc_time_cols:
                ax.set_yscale("log")
            # Use log scale for M (like reference image)
            if x_col == "n_subquantizers":
                ax.set_xscale("log")
                unique_subq = sorted(sub[x_col].unique())
                if len(unique_subq) > 4:
                    indices = np.linspace(0, len(unique_subq) - 1, 4, dtype=int)
                    tick_values = [unique_subq[i] for i in indices]
                else:
                    tick_values = unique_subq
                # Store tick values to reapply after tight_layout
                tick_values_to_use = tick_values
            elif x_col == "nbits":
                # Explicitly show every unique nbits value on the x-axis
                unique_nbits = sorted(sub[x_col].unique())
                tick_values_to_use = unique_nbits
                ax.set_xticks(unique_nbits)
                if y_col == "rel_error_mean":
                    # Slightly widen the Bps axis span for avg relative error vs Bps figures
                    ax.margins(x=0.08)

            # Grid styling like reference
            ax.grid(alpha=0.8, axis='y', linestyle='--')
            for spine in ax.spines.values():
                spine.set_visible(False)
            
            # Place legend outside plot when nbits or bits_per_vector is x-axis to avoid interference
            if x_col in ["nbits", "bits_per_vector"]:
                ax.legend(frameon=False, loc='center left', bbox_to_anchor=(1.05, 0.5))
            else:
                ax.legend(frameon=False, loc='best')
            
            # Apply tight_layout first
            plt.tight_layout()
            
            # Re-apply ticks for log scale after tight_layout to ensure they're preserved
            if x_col == "n_subquantizers" and tick_values_to_use is not None:
                ax.set_xticks(tick_values_to_use)
                ax.set_xticklabels([int(x) for x in tick_values_to_use])

            nudge_bits_per_subq_xlabel(ax)

            # Save figure with consistent DPI
            safe_method = method.lower()
            safe_dataset = dataset.lower()
            safe_x = x_col.replace("_", "")
            # Include y_col in filename if it's not the default
            if y_col != "rel_error_mean":
                safe_y = y_col.replace("_", "")
                savepath = output_dir / f"{safe_y}_{safe_method}_{safe_dataset}_{safe_x}.pdf"
            else:
                savepath = output_dir / f"relerr_{safe_method}_{safe_dataset}_{safe_x}.pdf"
            # fig.savefig(savepath, bbox_inches='tight', dpi=300)
            fig.savefig(savepath, dpi=300)
            
            print(f"Saved figure to {savepath}")
            
            plt.show()
            plt.close(fig)


def plot_bar_chart(
    df: pd.DataFrame,
    x_col: str,
    x_label: str,
    y_col: str = "train_time_s",
    y_label: str = None,
    methods: list = None,
    datasets: list = None,
    group_by: str = None,
    output_dir: Path = None
):
    """
    Plot bar chart (bin-style) for y_col vs x_col, with separate figures per method and dataset.
    
    Args:
        df: DataFrame with results
        x_col: Column name to plot on x-axis (will be grouped into bars)
        x_label: Display label for x-axis
        y_col: Column name to plot on y-axis (default: "train_time_s")
        y_label: Display label for y-axis (if None, auto-generated from y_col)
        methods: List of methods to plot (if None, uses all available)
        datasets: List of datasets to plot (if None, uses all available)
        group_by: "nbits" or "n_subquantizers" - how to group the bars (different bars for each group)
        output_dir: Directory to save plots
    """
    if methods is None:
        methods = sorted(df["method"].unique())
    if datasets is None:
        datasets = sorted(df["dataset"].unique())
    
    if output_dir is None:
        output_dir = Path("./../../experiments/plots/relerr_cpp")
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Determine grouping variable
    if group_by is None:
        # Default: group by the other variable (if x_col is nbits, group by n_subquantizers)
        if x_col == "nbits":
            group_col = "n_subquantizers"
        elif x_col == "n_subquantizers":
            group_col = "nbits"
        else:
            group_col = "nbits"
    elif group_by == "nbits":
        group_col = "nbits"
    elif group_by == "n_subquantizers":
        group_col = "n_subquantizers"
    else:
        raise ValueError(f"group_by must be 'nbits' or 'n_subquantizers', got {group_by}")
    
    for method in methods:
        for dataset in datasets:
            sub = df[(df["method"] == method) & (df["dataset"] == dataset)].copy()
            if sub.empty:
                continue
            
            # Create figure
            fig, ax = plt.subplots()
            
            # Get unique x_col and group_col values
            unique_x_vals = sorted(sub[x_col].unique())
            unique_group_vals = sorted(sub[group_col].unique())
            
            # Number of groups (x_col values) and bars per group (group_col values)
            n_groups = len(unique_x_vals)
            n_bars_per_group = len(unique_group_vals)
            
            # Set up bar positions for grouped bars
            bar_width = 0.8 / n_bars_per_group
            x_positions = np.arange(n_groups)
            
            # Color palette for different group_col values - same color for same group_col value
            colors = plt.cm.tab10(np.linspace(0, 1, n_bars_per_group))
            
            # Plot bars for each group_col value
            for i, group_val in enumerate(unique_group_vals):
                # Calculate y values for this group_col value across all x_col values
                y_vals = []
                for x_val in unique_x_vals:
                    # Get data for this specific x_col and group_col combination
                    data = sub[(sub[x_col] == x_val) & (sub[group_col] == group_val)]
                    if not data.empty:
                        # Use the actual value (not averaged) - if multiple rows exist, take mean
                        y_val = data[y_col].iloc[0] if len(data) == 1 else data[y_col].mean()
                        y_vals.append(y_val)
                        y_vals.append(0)
                
                # Calculate bar positions (offset for grouped bars)
                bar_positions = x_positions + i * bar_width
                
                # Plot bars - all bars with same group_col value get the same color
                ax.bar(
                    bar_positions,
                    y_vals,
                    width=bar_width,
                    label=curve_legend_label(group_col, group_val),
                    color=colors[i],
                    edgecolor="black",
                    linewidth=2,
                    alpha=0.8
                )
            
            # Set labels and styling
            ax.set_xlabel(x_label, fontsize=40)
            if y_label is None:
                if y_col == "train_time_s":
                    y_label = "Training time (seconds)"
                elif y_col == "adc_time_s":
                    y_label = "ADC time (seconds)"
                elif y_col == "distance_table_time_s":
                    y_label = "Distance table time (seconds)"
                    y_label = y_col.replace("_", " ").title()
            ax.set_ylabel(y_label, fontsize=40)
            ax.tick_params(labelsize=39)
            
            # Set x-axis ticks at the center of each group
            ax.set_xticks(x_positions)
            ax.set_xticklabels([int(x) if isinstance(x, (int, np.integer)) or (isinstance(x, float) and x.is_integer()) else x 
                               for x in unique_x_vals])
            
            # Add legend
            # ax.legend(frameon=False, fontsize=30)
            
            # Grid styling - match reference image style
            ax.grid(alpha=0.8, axis='y', linestyle='--')
            for spine in ax.spines.values():
                spine.set_visible(False)
            
            # Apply tight_layout
            plt.tight_layout()

            nudge_bits_per_subq_xlabel(ax)

            # Save figure
            safe_method = method.lower()
            safe_dataset = dataset.lower()
            safe_x = x_col.replace("_", "")
            safe_y = y_col.replace("_", "")
            savepath = output_dir / f"bar_{safe_y}_{safe_method}_{safe_dataset}_{safe_x}.pdf"
            # fig.savefig(savepath, bbox_inches='tight', dpi=300)
            fig.savefig(savepath, dpi=300)
            print(f"Saved figure to {savepath}")
            
            plt.show()
            plt.close(fig)


def plot_compression_rate_vs_y(
    df: pd.DataFrame,
    y_col: str,
    y_label: str,
    methods: list = None,
    datasets: list = None,
    group_by: str = "nbits",
    output_dir: Path = None,
    nbits_subquantizers=None,
    num_subq_plot_subquantizers=None,
):
    """
    Plot chosen metric vs compression rate, with separate figures per method and dataset.
    X-axis: compression_rate, Y-axis: y_col.
    Curves are grouped by group_by ("nbits" or "n_subquantizers").
    """
    if methods is None:
        methods = sorted(df["method"].unique())
    if datasets is None:
        datasets = sorted(df["dataset"].unique())

    if output_dir is None:
        output_dir = PQ_FAISS_FIGURES_DIR
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    if y_col not in df.columns:
        return
    group_col = "nbits" if group_by == "nbits" else "n_subquantizers"
    safe_y = y_col.replace("_", "")

    for method in methods:
        for dataset in datasets:
            sub = df[(df["method"] == method) & (df["dataset"] == dataset)].copy()
            if sub.empty:
                continue
            sub = sub.dropna(subset=["compression_rate", y_col])
            if sub.empty:
                continue

            if nbits_subquantizers is not None:
                subq_list = nbits_subquantizers.get(dataset) if isinstance(nbits_subquantizers, dict) else nbits_subquantizers
                if subq_list is not None:
                    sub = sub[sub["n_subquantizers"].isin(subq_list)]
            if num_subq_plot_subquantizers is not None:
                subq_list = num_subq_plot_subquantizers.get(dataset) if isinstance(num_subq_plot_subquantizers, dict) else num_subq_plot_subquantizers
                if subq_list is not None:
                    sub = sub[sub["n_subquantizers"].isin(subq_list)]
            if sub.empty:
                continue

            unique_group_values = sorted(sub[group_col].unique())
            color_map = create_dynamic_color_map(unique_group_values)
            marker_map = create_dynamic_marker_map(unique_group_values)

            fig, ax = plt.subplots()
            grouped = sub.groupby(group_col)
            for group_val, group_df in grouped:
                group_df_sorted = group_df.sort_values("compression_rate")
                color = color_map[group_val]
                marker = marker_map[group_val]
                ax.plot(
                    group_df_sorted["compression_rate"],
                    group_df_sorted[y_col],
                    label=curve_legend_label(group_col, group_val),
                    color=color,
                    marker=marker,
                    markersize=12,
                    linewidth=2,
                    markeredgewidth=2,
                    markeredgecolor="black",
                )

            ax.set_xlabel("Compression rate", fontsize=40)
            ax.set_ylabel(y_label, fontsize=40)
            ax.tick_params(labelsize=39)
            ax.xaxis.set_major_locator(MaxNLocator(nbins=5))
            ax.grid(alpha=0.8, axis="y", linestyle="--")
            for spine in ax.spines.values():
                spine.set_visible(False)
            ax.legend(frameon=False, loc="center left", bbox_to_anchor=(1.05, 0.5))
            plt.tight_layout()

            safe_method = method.lower()
            safe_dataset = dataset.lower()
            savepath = output_dir / f"compression_rate_vs_{safe_y}_{safe_method}_{safe_dataset}.pdf"
            fig.savefig(savepath, dpi=300)
            print(f"Saved figure to {savepath}")
            plt.show()
            plt.close(fig)


def plot_opq_vs_pq_rot_percent(
    df: pd.DataFrame,
    dataset: str,
    nbits_list,
    n_subquantizers_list,
    y_col: str = "rel_error_mean",
    y_label: str = None,
    higher_is_better: bool = False,
    output_dir: Path = None,
    figsize: tuple = (10, 6),
    n_bins: int = 20,
):
    """
    Plot y_col vs % of (max_opq_rot_train_samples / rot_train_sz), averaged across
    all (nbits, n_subquantizers) in nbits_list x n_subquantizers_list.
    PQ: horizontal line = mean y_col over those pairs.
    OPQ: curve = for each x (pct_rot bin), mean y_col over all selected pairs in that bin.
    Plot PQ and OPQ as lines without shaded comparison bands.
    nbits_list and n_subquantizers_list can be lists or single ints (converted to list).
    """
    if output_dir is None:
        output_dir = PQ_FAISS_FIGURES_DIR
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    nbits_list = [nbits_list] if np.isscalar(nbits_list) else list(nbits_list)
    n_subquantizers_list = [n_subquantizers_list] if np.isscalar(n_subquantizers_list) else list(n_subquantizers_list)

    for col in ["rot_train_sz", "max_opq_rot_train_samples"]:
        if col not in df.columns:
            raise ValueError(
                f"DataFrame must contain OPQ columns 'rot_train_sz' and 'max_opq_rot_train_samples'. Missing: {col}"
            )

    # PQ: average rel_error over all (dataset, nbits, n_subquantizers) in the lists
    pq = df[(df["method"] == "PQ") & (df["dataset"] == dataset) & (df["nbits"].isin(nbits_list)) & (df["n_subquantizers"].isin(n_subquantizers_list))]
    if pq.empty:
        raise ValueError(f"No PQ data for dataset={dataset}, nbits in {nbits_list}, n_subquantizers in {n_subquantizers_list}")
    if y_col not in pq.columns:
        raise ValueError(f"Column '{y_col}' not found in dataframe")
    pq_val = float(pq[y_col].mean())

    # OPQ: all rows for (dataset, nbits, n_subquantizers) in the lists; compute pct_rot
    opq = df[(df["method"] == "OPQ") & (df["dataset"] == dataset) & (df["nbits"].isin(nbits_list)) & (df["n_subquantizers"].isin(n_subquantizers_list))].copy()
    opq = opq.dropna(subset=["rot_train_sz", "max_opq_rot_train_samples", y_col])
    if opq.empty:
        raise ValueError(f"No OPQ data for dataset={dataset}, nbits in {nbits_list}, n_subquantizers in {n_subquantizers_list}")
    opq["pct_rot"] = 100.0 * opq["max_opq_rot_train_samples"] / opq["rot_train_sz"]

    # Bin pct_rot and average y_col in each bin (one curve across all (nbits, M))
    pct_min, pct_max = opq["pct_rot"].min(), opq["pct_rot"].max()
    if pct_min >= pct_max or np.isclose(pct_min, pct_max):
        # All OPQ runs used same pct (e.g. 100%); single point.
        x = np.array([pct_min])
        y_opq = np.array([opq[y_col].mean()])
    else:
        bins = np.linspace(pct_min, pct_max, n_bins + 1)
        opq["_bin"] = pd.cut(opq["pct_rot"], bins=bins, include_lowest=True)
        agg = opq.groupby("_bin", observed=True).agg({"pct_rot": "mean", y_col: "mean"}).reset_index()
        agg = agg.dropna(subset=["pct_rot", y_col])
        if agg.empty:
            raise ValueError(f"No OPQ bins for dataset={dataset}, metric={y_col} after pct_rot binning")
        x = agg["pct_rot"].values
        y_opq = agg[y_col].values
    y_pq = np.full_like(x, pq_val)

    fig, ax = plt.subplots(figsize=figsize)
    # Custom markers and colors from notebook palette (consistent with other figures)
    ax.plot(x, y_pq, color=COLOR_PALETTE[0], linewidth=2, marker="o", markersize=12, markeredgecolor="black", markeredgewidth=2, label="PQ")
    ax.plot(x, y_opq, color=COLOR_PALETTE[1], linewidth=2, marker="s", markersize=12, markeredgecolor="black", markeredgewidth=2, label="OPQ")
    ax.set_xticks(x)
    ax.set_xlabel("% data used to learn R", fontsize=40)
    if y_label is None:
        y_label = y_col.replace("_", " ").title()
    ax.set_ylabel(y_label, fontsize=40)
    ax.tick_params(labelsize=39)
    # Widen y-axis for a better view (add ~15% padding above and below data range)
    y_lo = min(pq_val, y_opq.min())
    y_hi = max(pq_val, y_opq.max())
    pad = max((y_hi - y_lo) * 0.15, 0.005)
    ax.set_ylim(y_lo - pad, y_hi + pad)
    ax.grid(alpha=0.8, axis="y", linestyle="--")
    for spine in ax.spines.values():
        spine.set_visible(False)
    # Keep PQ/OPQ legend horizontal and out of the data region.
    ax.legend(
        frameon=False,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.18),
        ncol=2,
        fontsize=21,
    )
    plt.tight_layout(rect=(0, 0, 1, 0.93))
    suffix = f"M{min(n_subquantizers_list)}-{max(n_subquantizers_list)}_nbits{min(nbits_list)}-{max(nbits_list)}" if (len(n_subquantizers_list) > 1 or len(nbits_list) > 1) else f"M{n_subquantizers_list[0]}_nbits{nbits_list[0]}"
    safe_y = y_col.replace("_", "")
    savepath = output_dir / f"opq_vs_pq_rot_pct_{safe_y}_{dataset}_{suffix}.pdf"
    fig.savefig(savepath, dpi=300)
    print(f"Saved {savepath}")
    plt.show()
    plt.close(fig)


def plot_opq_vs_pq_train_time_rot_percent(
    df: pd.DataFrame,
    dataset: str,
    nbits_list,
    n_subquantizers_list,
    output_dir: Path = None,
    figsize: tuple = (10, 6),
    n_bins: int = 20,
):
    """
    Plot training time vs % of data used to learn R, averaged across
    all (nbits, n_subquantizers) in nbits_list x n_subquantizers_list.
    PQ: horizontal line = mean train_time_s over those pairs.
    OPQ: curve = train_time_s + opq_train_time_s per run, binned by pct_rot and averaged.
    Same x-axis and averaging logic as plot_opq_vs_pq_rot_percent.
    """
    if output_dir is None:
        output_dir = PQ_FAISS_FIGURES_DIR
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    nbits_list = [nbits_list] if np.isscalar(nbits_list) else list(nbits_list)
    n_subquantizers_list = [n_subquantizers_list] if np.isscalar(n_subquantizers_list) else list(n_subquantizers_list)

    for col in ["rot_train_sz", "max_opq_rot_train_samples", "train_time_s"]:
        if col not in df.columns:
            raise ValueError(f"DataFrame must contain '{col}'. Missing: {col}")

    # PQ: average train_time_s over all (dataset, nbits, n_subquantizers) in the lists
    pq = df[(df["method"] == "PQ") & (df["dataset"] == dataset) & (df["nbits"].isin(nbits_list)) & (df["n_subquantizers"].isin(n_subquantizers_list))]
    if pq.empty:
        raise ValueError(f"No PQ data for dataset={dataset}, nbits in {nbits_list}, n_subquantizers in {n_subquantizers_list}")
    pq_time = float(pq["train_time_s"].mean())

    # OPQ: train_time_s + opq_train_time_s (use train_time_s only if opq_train_time_s missing)
    opq = df[(df["method"] == "OPQ") & (df["dataset"] == dataset) & (df["nbits"].isin(nbits_list)) & (df["n_subquantizers"].isin(n_subquantizers_list))].copy()
    opq = opq.dropna(subset=["rot_train_sz", "max_opq_rot_train_samples", "train_time_s"])
    if opq.empty:
        raise ValueError(f"No OPQ data for dataset={dataset}, nbits in {nbits_list}, n_subquantizers in {n_subquantizers_list}")
    if "opq_train_time_s" in opq.columns:
        opq["total_train_time_s"] = opq["train_time_s"] + opq["opq_train_time_s"].fillna(0)
    else:
        opq["total_train_time_s"] = opq["train_time_s"]
    opq["pct_rot"] = 100.0 * opq["max_opq_rot_train_samples"] / opq["rot_train_sz"]

    # Bin pct_rot and average total_train_time_s in each bin.
    pct_min, pct_max = opq["pct_rot"].min(), opq["pct_rot"].max()
    if pct_min >= pct_max or np.isclose(pct_min, pct_max):
        x = np.array([pct_min])
        y_opq = np.array([opq["total_train_time_s"].mean() / 60])
    else:
        bins = np.linspace(pct_min, pct_max, n_bins + 1)
        opq["_bin"] = pd.cut(opq["pct_rot"], bins=bins, include_lowest=True)
        agg = opq.groupby("_bin", observed=True).agg({"pct_rot": "mean", "total_train_time_s": "mean"}).reset_index()
        agg = agg.dropna(subset=["pct_rot", "total_train_time_s"])
        if agg.empty:
            raise ValueError(f"No OPQ train-time bins for dataset={dataset} after pct_rot binning")
        x = agg["pct_rot"].values
        y_opq = agg["total_train_time_s"].values / 60
    pq_time = pq_time / 60  # convert to minutes
    y_pq = np.full_like(x, pq_time)

    fig, ax = plt.subplots(figsize=figsize)
    ax.plot(x, y_pq, color=COLOR_PALETTE[0], linewidth=2, marker="o", markersize=12, markeredgecolor="black", markeredgewidth=2, label="PQ")
    ax.plot(x, y_opq, color=COLOR_PALETTE[1], linewidth=2, marker="s", markersize=12, markeredgecolor="black", markeredgewidth=2, label="OPQ")
    ax.set_xticks(x)
    ax.set_xlabel("% data used to learn R", fontsize=40)
    ax.set_ylabel("Train time (m)", fontsize=40)
    ax.tick_params(labelsize=39)
    y_lo = min(pq_time, y_opq.min())
    y_hi = max(pq_time, y_opq.max())
    pad = max((y_hi - y_lo) * 0.15, 0.05)
    ax.set_ylim(y_lo - pad, y_hi + pad)
    # Exactly 3 y-ticks: beginning, middle, end
    ax.set_yticks([y_lo, (y_lo + y_hi) / 2, y_hi])
    ax.grid(alpha=0.8, axis="y", linestyle="--")
    for spine in ax.spines.values():
        spine.set_visible(False)
    # Keep PQ/OPQ legend horizontal and out of the data region.
    ax.legend(
        frameon=False,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.18),
        ncol=2,
        fontsize=21,
    )
    plt.tight_layout(rect=(0, 0, 1, 0.93))
    suffix = f"M{min(n_subquantizers_list)}-{max(n_subquantizers_list)}_nbits{min(nbits_list)}-{max(nbits_list)}" if (len(n_subquantizers_list) > 1 or len(nbits_list) > 1) else f"M{n_subquantizers_list[0]}_nbits{nbits_list[0]}"
    savepath = output_dir / f"opq_vs_pq_train_time_rot_pct_{dataset}_{suffix}.pdf"
    fig.savefig(savepath, dpi=300)
    print(f"Saved {savepath}")
    plt.show()
    plt.close(fig)



# ----- Single (y-metric, x-column) for this split notebook -----
Y_METRICS = [TARGET_Y_METRIC]
_adc_xcols = ("adc_cpu_time_pp", "adc_cpu_time_pp_ms")
_x_pin = [t for t in X_COLUMNS_TO_PLOT if t[0] == TARGET_X_COL]
if len(_x_pin) != 1 and TARGET_X_COL in _adc_xcols:
    _x_pin = [t for t in X_COLUMNS_TO_PLOT if t[0] in _adc_xcols]
if len(_x_pin) != 1:
    raise ValueError(
        "TARGET_X_COL=%r must match exactly one entry in X_COLUMNS_TO_PLOT "
        "(after USER_* overrides); got columns %r"
        % (TARGET_X_COL, [t[0] for t in X_COLUMNS_TO_PLOT])
    )
X_COLUMNS_TO_PLOT = _x_pin
del _x_pin, _adc_xcols

# Generate plots based on configuration (for each Y_METRIC: relerr vs nbits, vs M, etc.)
metrics_to_plot = []
for m in Y_METRICS:
    if m == "recall":
        metrics_to_plot.extend(["recall_1", "recall_10", "recall_100"])
    else:
        metrics_to_plot.append(m)

for x_col, x_label in X_COLUMNS_TO_PLOT:
    if x_col not in plot_df.columns:
        print(f"Warning: Column '{x_col}' not found in dataframe. Skipping...")
        continue

    ylim = YLIM_CONFIG.get(x_col, None)

    for metric in metrics_to_plot:
        if metric not in Y_METRIC_MAP:
            continue
        y_col, y_label = Y_METRIC_MAP[metric]
        if y_col not in plot_df.columns:
            print(
                f"Warning: y column {y_col!r} (metric {metric!r}) not in plot_df — "
                f"merge eval CSVs under DATA_DIR (e.g. *_reconstruction_error.csv). Skipping."
            )
            continue
        if not plot_df[y_col].notna().any():
            print(
                f"Warning: y column {y_col!r} is all null — populate it (e.g. "
                f"`python scripts/evals/run_evals.py` with reconstruction_error). "
                f"Skipping {metric!r} vs {x_label!r}."
            )
            continue

        print(f"\n{'='*60}")
        print(f"Plotting: {y_label} vs {x_label}")
        print(f"Methods: {METHODS_TO_PLOT}, Datasets: {DATASETS_TO_PLOT}")
        if ylim is not None:
            print(f"Y-axis limits: {ylim}")
        print(f"{'='*60}\n")
        plot_relerr_vs_x(
            plot_df,
            x_col,
            x_label,
            y_col=y_col,
            y_label=y_label,
            methods=METHODS_TO_PLOT,
            datasets=DATASETS_TO_PLOT,
            ylim=ylim,
            output_dir=PQ_FAISS_FIGURES_DIR,
            nbits_subquantizers=NBITS_PLOT_SUBQUANTIZERS,
            num_subq_plot_subquantizers=NUM_SUBQ_PLOT_SUBQUANTIZERS,
        )

